# 01 Data Preprocessing

This notebook loads the Student Dropout dataset, inspects its structure, handles missing and non-numeric values, and standardizes features to prepare for downstream tasks (t-SNE, clustering, prediction).

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/data.csv', sep=';')
print(f'Shape: {df.shape}')
df.head()

Shape: (4424, 37)


,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 1. Basic Inspection

We first examine data types, missing values, and duplicates to understand the dataset's overall quality.

In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

In [3]:
df.describe()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
count,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,...,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000,4424.000000
mean,1.178571,18.669078,1.727848,8856.642631,0.890823,4.577758,132.613314,1.873192,19.561935,22.275316,...,0.137658,0.541817,6.232143,8.063291,4.435805,10.230206,0.150316,11.566139,1.228029,0.001969
std,0.605747,17.484682,1.313793,2063.566416,0.311897,10.216592,13.188332,6.914514,15.603186,15.343108,...,0.690880,1.918546,2.195951,3.947951,3.014764,5.210808,0.753774,2.663850,1.382711,2.269935
min,1.000000,1.000000,0.000000,33.000000,0.000000,1.000000,95.000000,1.000000,1.000000,1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,7.600000,-0.800000,-4.060000
25%,1.000000,1.000000,1.000000,9085.000000,1.000000,1.000000,125.000000,1.000000,2.000000,3.000000,...,0.000000,0.000000,5.000000,6.000000,2.000000,10.750000,0.000000,9.400000,0.300000,-1.700000
50%,1.000000,17.000000,1.000000,9238.000000,1.000000,1.000000,133.100000,1.000000,19.000000,19.000000,...,0.000000,0.000000,6.000000,8.000000,5.000000,12.200000,0.000000,11.100000,1.400000,0.320000
75%,1.000000,39.000000,2.000000,9556.000000,1.000000,1.000000,140.000000,1.000000,37.000000,37.000000,...,0.000000,0.000000,7.000000,10.000000,6.000000,13.333333,0.000000,13.900000,2.600000,1.790000
max,6.000000,57.000000,9.000000,9991.000000,1.000000,43.000000,190.000000,109.000000,44.000000,44.000000,...,12.000000,19.000000,23.000000,33.000000,20.000000,18.571429,12.000000,16.200000,3.700000,3.510000


In [4]:
# Check missing values
missing = df.isnull().sum()
print('Columns with missing values:')
print(missing[missing > 0] if missing.any() else 'None — dataset has no missing values.')

Columns with missing values:
None — dataset has no missing values.


In [5]:
# Check duplicate rows
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup}')
if n_dup > 0:
    df = df.drop_duplicates()
    print(f'After removal: {df.shape}')

Duplicate rows: 0


In [6]:
# Target distribution
print('Target distribution:')
print(df['Target'].value_counts())
print(f'\nPercentages:')
print(df['Target'].value_counts(normalize=True).round(3))

Target distribution:
Target
Graduate    2209
Dropout     1421
Enrolled     794
Name: count, dtype: int64

Percentages:
Target
Graduate    0.499
Dropout     0.321
Enrolled    0.179
Name: proportion, dtype: float64


### Observations

- **No missing values** across all 37 columns — no imputation needed.
- **No duplicate rows.**
- **Class imbalance**: Graduate (~50%) is the majority class, followed by Dropout (~32%) and Enrolled (~18%). This imbalance should be considered during model training and evaluation.
- All features except `Target` are already numeric (int64 or float64), but many integer columns are actually **categorical codes** (e.g., `Course`, `Nacionality`).

## 2. Column Name Cleanup

Some column names contain trailing tabs and quotes from the CSV. We strip these for consistency.

In [7]:
df.columns = df.columns.str.strip().str.strip('"').str.strip()
df.columns.tolist()

['Marital status',
 'Application mode',
 'Application order',
 'Course',
 'Daytime/evening attendance',
 'Previous qualification',
 'Previous qualification (grade)',
 'Nacionality',
 "Mother's qualification",
 "Father's qualification",
 "Mother's occupation",
 "Father's occupation",
 'Admission grade',
 'Displaced',
 'Educational special needs',
 'Debtor',
 'Tuition fees up to date',
 'Gender',
 'Scholarship holder',
 'Age at enrollment',
 'International',
 'Curricular units 1st sem (credited)',
 'Curricular units 1st sem (enrolled)',
 'Curricular units 1st sem (evaluations)',
 'Curricular units 1st sem (approved)',
 'Curricular units 1st sem (grade)',
 'Curricular units 1st sem (without evaluations)',
 'Curricular units 2nd sem (credited)',
 'Curricular units 2nd sem (enrolled)',
 'Curricular units 2nd sem (evaluations)',
 'Curricular units 2nd sem (approved)',
 'Curricular units 2nd sem (grade)',
 'Curricular units 2nd sem (without evaluations)',
 'Unemployment rate',
 'Inflation rat

## 3. Identify Feature Types

Although all features are numeric, many are integer-coded categorical variables. We classify columns into three groups:
- **Binary**: already 0/1 (e.g., Gender, Displaced)
- **Nominal categorical**: integer codes with no ordinal meaning (e.g., Course, Nacionality)
- **Continuous numeric**: truly numeric features (e.g., grades, age)

In [8]:
# Columns with few unique values
print('Columns with <= 20 unique values (likely categorical):\n')
for col in df.columns:
    n_unique = df[col].nunique()
    if n_unique <= 20:
        print(f'  {col}: {n_unique} unique')

Columns with <= 20 unique values (likely categorical):

  Marital status: 6 unique
  Application mode: 18 unique
  Application order: 8 unique
  Course: 17 unique
  Daytime/evening attendance: 2 unique
  Previous qualification: 17 unique
  Displaced: 2 unique
  Educational special needs: 2 unique
  Debtor: 2 unique
  Tuition fees up to date: 2 unique
  Gender: 2 unique
  Scholarship holder: 2 unique
  International: 2 unique
  Curricular units 1st sem (without evaluations): 11 unique
  Curricular units 2nd sem (credited): 19 unique
  Curricular units 2nd sem (approved): 20 unique
  Curricular units 2nd sem (without evaluations): 10 unique
  Unemployment rate: 10 unique
  Inflation rate: 9 unique
  GDP: 10 unique
  Target: 3 unique


In [9]:
# Binary columns (already 0/1)
binary_cols = [col for col in df.columns if set(df[col].unique()) <= {0, 1}]
print(f'Binary columns ({len(binary_cols)}):', binary_cols)

# Nominal categorical columns (integer-coded, no ordinal meaning)
nominal_cols = [
    'Marital status', 'Application mode', 'Course',
    'Nacionality', "Mother's qualification", "Father's qualification",
    "Mother's occupation", "Father's occupation",
    'Previous qualification'
]
print(f'\nNominal categorical columns ({len(nominal_cols)}):', nominal_cols)

# Continuous numeric columns (everything else except Target)
numeric_cols = [col for col in df.columns
                if col not in binary_cols + nominal_cols + ['Target']]
print(f'\nContinuous numeric columns ({len(numeric_cols)}):', numeric_cols)

Binary columns (8): ['Daytime/evening attendance', 'Displaced', 'Educational special needs', 'Debtor', 'Tuition fees up to date', 'Gender', 'Scholarship holder', 'International']

Nominal categorical columns (9): ['Marital status', 'Application mode', 'Course', 'Nacionality', "Mother's qualification", "Father's qualification", "Mother's occupation", "Father's occupation", 'Previous qualification']

Continuous numeric columns (19): ['Application order', 'Previous qualification (grade)', 'Admission grade', 'Age at enrollment', 'Curricular units 1st sem (credited)', 'Curricular units 1st sem (enrolled)', 'Curricular units 1st sem (evaluations)', 'Curricular units 1st sem (approved)', 'Curricular units 1st sem (grade)', 'Curricular units 1st sem (without evaluations)', 'Curricular units 2nd sem (credited)', 'Curricular units 2nd sem (enrolled)', 'Curricular units 2nd sem (evaluations)', 'Curricular units 2nd sem (approved)', 'Curricular units 2nd sem (grade)', 'Curricular units 2nd sem (wi

## 4. Handle Non-Numeric Values

The `Target` column is the only non-numeric (object) column. We encode it using Label Encoding for model training.

For the **nominal categorical columns** (integer-coded but with no ordinal relationship), we apply **one-hot encoding** so that models do not incorrectly assume ordering between categories. To keep dimensionality manageable, we use `drop_first=True` to avoid multicollinearity.

In [10]:
from sklearn.preprocessing import LabelEncoder

# Encode target variable
le = LabelEncoder()
df['Target_encoded'] = le.fit_transform(df['Target'])
print('Target encoding:', dict(zip(le.classes_, le.transform(le.classes_))))

Target encoding: {'Dropout': np.int64(0), 'Enrolled': np.int64(1), 'Graduate': np.int64(2)}


In [11]:
# One-hot encode nominal categorical columns
print(f'Shape before one-hot encoding: {df.shape}')

df_encoded = pd.get_dummies(df, columns=nominal_cols, drop_first=True)

print(f'Shape after one-hot encoding:  {df_encoded.shape}')
print(f'\nNew columns added: {df_encoded.shape[1] - df.shape[1]}')

Shape before one-hot encoding: (4424, 38)
Shape after one-hot encoding:  (4424, 240)

New columns added: 202


## 5. Outlier Detection

We use the IQR method to identify outliers in continuous features. We report them here for awareness but **do not remove** them at this stage — many "outliers" (e.g., 0 grades for dropouts) are valid data points. Removal decisions should be made during modeling based on impact.

In [12]:
print('IQR-based outlier counts for continuous features:\n')
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = ((df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)).sum()
    if outliers > 0:
        print(f'  {col}: {outliers} outliers ({outliers/len(df)*100:.1f}%)')

IQR-based outlier counts for continuous features:

  Application order: 541 outliers (12.2%)
  Previous qualification (grade): 179 outliers (4.0%)
  Admission grade: 86 outliers (1.9%)
  Age at enrollment: 441 outliers (10.0%)
  Curricular units 1st sem (credited): 577 outliers (13.0%)
  Curricular units 1st sem (enrolled): 424 outliers (9.6%)
  Curricular units 1st sem (evaluations): 158 outliers (3.6%)
  Curricular units 1st sem (approved): 180 outliers (4.1%)
  Curricular units 1st sem (grade): 726 outliers (16.4%)
  Curricular units 1st sem (without evaluations): 294 outliers (6.6%)
  Curricular units 2nd sem (credited): 530 outliers (12.0%)
  Curricular units 2nd sem (enrolled): 369 outliers (8.3%)
  Curricular units 2nd sem (evaluations): 109 outliers (2.5%)
  Curricular units 2nd sem (approved): 44 outliers (1.0%)
  Curricular units 2nd sem (grade): 877 outliers (19.8%)
  Curricular units 2nd sem (without evaluations): 282 outliers (6.4%)


### Observations

- **Curricular unit grades** (1st & 2nd sem) have the most outliers (16-20%) — this is expected because dropout students often have 0 grades, which fall far below the IQR.
- **Age at enrollment** has ~10% outliers, reflecting mature/non-traditional students.
- **Credited units** (1st & 2nd sem) have ~12-13% outliers, corresponding to students with transfer credits.

## 6. Feature Standardization

We apply **StandardScaler** (zero mean, unit variance) to all continuous numeric features. This is important for:
- **t-SNE** (Phase 2): sensitive to feature scales
- **Clustering** (Phase 3): distance-based algorithms like K-means require scaled features
- **Models** (Phase 4): logistic regression and other algorithms benefit from standardization

Binary columns are left as-is (already 0/1). One-hot encoded columns are also left as-is (already 0/1).

In [13]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df_encoded[numeric_cols] = scaler.fit_transform(df_encoded[numeric_cols])

print('After standardization (continuous features):')
df_encoded[numeric_cols].describe().loc[['mean', 'std']].round(4)

After standardization (continuous features):


,Application order,Previous qualification (grade),Admission grade,Age at enrollment,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP
mean,-0.0000,-0.0000,-0.0000,-0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,-0.0000,0.0000,-0.0000,-0.0000,-0.0000,0.0000,-0.0000,0.0000,0.0000
std,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001,1.0001


## 7. Save Preprocessed Data

We save two versions:
- `data_cleaned.csv`: column names cleaned, target encoded, but **no** one-hot encoding or scaling (for EDA/visualization)
- `data_preprocessed.csv`: fully preprocessed with one-hot encoding and standardization (for clustering and modeling)

In [14]:
# Cleaned version (no encoding/scaling, useful for EDA)
df.to_csv('../data/data_cleaned.csv', index=False)
print(f'Saved data_cleaned.csv: {df.shape}')

# Fully preprocessed version (one-hot encoded + standardized)
df_encoded.to_csv('../data/data_preprocessed.csv', index=False)
print(f'Saved data_preprocessed.csv: {df_encoded.shape}')

Saved data_cleaned.csv: (4424, 38)
Saved data_preprocessed.csv: (4424, 240)


## Summary

| Step | Method | Result |
|------|--------|--------|
| Missing values | Checked with `isnull()` | None found |
| Duplicates | Checked with `duplicated()` | None found |
| Column cleanup | Strip whitespace/tabs | Fixed `Daytime/evening attendance` |
| Non-numeric handling | LabelEncoder for Target, one-hot encoding (`pd.get_dummies`) for 9 nominal columns | Expanded from 37 to 240 columns |
| Standardization | StandardScaler on 19 continuous features | Zero mean, unit variance |
| Outliers | IQR detection | Reported but not removed (valid data points) |

**Key patterns observed:**
- The dataset is clean with no missing values or duplicates.
- Class imbalance exists: Graduate ~50%, Dropout ~32%, Enrolled ~18%.
- Most features are integer-coded categories rather than true numeric values.
- High outlier counts in grade columns are driven by dropout students with zero grades, not data errors.